#### Librerías

In [1]:
import requests
import time
import pandas as pd
import json

#### Lista de 30 artistas populares (representando los mercados de Argentina, España y EEUU)

- Los IDs fueron obtenidos a través de la API de Deezer pero la función de búsqueda "search" demostró ser inconsiste, arrojando resultados incorrectos para ciertos artistas. Por ello, hicimos comprobaciones de los IDs de los 30 artistias, aplicando las correcciones necesarias usando como fuente Deezer.com.

- Uso de una lista de diccionarios ya que es más versátil y permite añadir más campos si fuese necesario más adelante

In [2]:
artists = [
    {"name": "Taylor Swift", "id": 12246},
    {"name": "Drake", "id": 246791},
    {"name": "Morgan Wallen", "id": 7188840},
    {"name": "Kendrick Lamar", "id": 525046},
    {"name": "The Weeknd", "id": 4050205},
    {"name": "SZA", "id": 5531258},
    {"name": "Zach Bryan", "id": 71855892},
    {"name": "Tyler, The Creator", "id": 1194083},
    {"name": "Billie Eilish", "id": 9635624},
    {"name": "Ariana Grande", "id": 1562681},

    {"name": "Bad Bunny", "id": 10583405},
    {"name": "Karol G", "id": 5297021},
    {"name": "Myke Towers", "id": 12029862},
    {"name": "Quevedo", "id": 6705223},
    {"name": "Aitana", "id": 11928391},
    {"name": "Rosalía", "id": 554792},
    {"name": "Morad", "id": 111130212},
    {"name": "Melendi", "id": 2697},
    {"name": "Enrique Iglesias", "id": 2792},
    {"name": "Juanes", "id": 70},

    {"name": "Duki", "id": 4902904},
    {"name": "María Becerra", "id": 14343187},
    {"name": "Tini Stoessel", "id": 2252591},
    {"name": "Nicki Nicole", "id": 13299379},
    {"name": "Cazzu", "id": 12268072},
    {"name": "Lali", "id": 5181388},
    {"name": "Paulo Londra", "id": 12637039},
    {"name": "Shakira", "id": 160},
    {"name": "Bizarrap", "id": 12487862},
    {"name": "Emilia", "id": 61094422}
]

### Búsqueda del top 50 de canciones de los artistas elegidos 

La búsqueda se lleva a cabo mediante el endpoint indicado "hhttps://api.deezer.com/artist/{id}/top?limit=50"

In [3]:
songs = []

for artist in artists:
    artist_id = artist["id"]
    artist_name = artist["name"]

    tracks_request = requests.get(f"https://api.deezer.com/artist/{artist_id}/top?limit=50")

    if tracks_request.status_code != 200:
        print(f"Could not fetch tracks for artist: {artist_name}")
        continue

    tracks_data = tracks_request.json()

    for track in tracks_data["data"]:
        songs.append({
            "artist_id": artist_id,
            "artist_name": artist_name,
            "track_title": track["title"],
            "album_title": track["album"]["title"],
            "track_type": track["type"]
        })
        
    time.sleep(1)

#### Comprobación del número de canciones recogidas para cada artista 

No todos los artistas tienen 50 canciones en el top de Deezer ya que artistas regionales menos populares a nivel global cuentan con un número menor de canciones en esta recopilación concreta de canciones, se trata de una limitación de la propia API. Comprobamos que el número indicado para cada artista se corresponde con la info en Deezer.com

In [4]:
song_count = {}

for artist in artists:
    artist_id = artist["id"]
    artist_name = artist["name"]

    url = f"https://api.deezer.com/artist/{artist_id}/top?limit=50"
    response = requests.get(url)

    if response.status_code != 200:
        song_count[artist_name] = "Error"
        continue

    data = response.json()
    num_songs = len(data.get("data", []))

    song_count[artist_name] = num_songs

    time.sleep(1)

song_count

{'Taylor Swift': 50,
 'Drake': 50,
 'Morgan Wallen': 50,
 'Kendrick Lamar': 50,
 'The Weeknd': 50,
 'SZA': 50,
 'Zach Bryan': 50,
 'Tyler, The Creator': 50,
 'Billie Eilish': 50,
 'Ariana Grande': 50,
 'Bad Bunny': 50,
 'Karol G': 50,
 'Myke Towers': 50,
 'Quevedo': 50,
 'Aitana': 50,
 'Rosalía': 50,
 'Morad': 50,
 'Melendi': 50,
 'Enrique Iglesias': 50,
 'Juanes': 50,
 'Duki': 50,
 'María Becerra': 50,
 'Tini Stoessel': 50,
 'Nicki Nicole': 50,
 'Cazzu': 50,
 'Lali': 50,
 'Paulo Londra': 50,
 'Shakira': 50,
 'Bizarrap': 50,
 'Emilia': 50}

In [5]:
songs = []
album_cache = {}

for i, artist in enumerate(artists, 1):
    artist_id = artist["id"]
    artist_name = artist["name"]
    
    print(f"[{i}/{len(artists)}] Processing: {artist_name}")
    
    tracks_request = requests.get(f"https://api.deezer.com/artist/{artist_id}/top?limit=50")
    if tracks_request.status_code != 200:
        print(f"  Error fetching tracks for {artist_name}")
        continue
    
    tracks_data = tracks_request.json()
    
    for track in tracks_data["data"]:
        album_id = track["album"]["id"]
        
        if album_id not in album_cache:
            album_request = requests.get(f"https://api.deezer.com/album/{album_id}")
            album_cache[album_id] = album_request.json() if album_request.status_code == 200 else {}
            time.sleep(0.3)
        
        album_data = album_cache[album_id]
        
        genre_list   = album_data.get("genres", {}).get("data", [])
        genre        = genre_list[0].get("name") if genre_list else None
        genre_id     = genre_list[0].get("id") if genre_list else None
        release_year = album_data.get("release_date", "")[:4] or None

        contributors = track.get("contributors", [])
        track_type   = "collab" if len(contributors) > 1 else "solo"

        songs.append({
            "artist_id":    artist_id,
            "artist_name":  artist_name,
            "track_title":  track["title"],
            "album_title":  track["album"]["title"],
            "track_type":   track_type,
            "release_year": release_year,
            "genre":        genre,
            "genre_id":     genre_id
        })
    
    time.sleep(1)

df_songs = pd.DataFrame(songs)

# Rellenar nulos
df_songs[["genre", "genre_id"]] = df_songs[["genre", "genre_id"]].fillna("Unknown")

# Limpiar el .0 del genre_id
df_songs["genre_id"] = df_songs["genre_id"].astype(str).str.replace(".0", "", regex=False)

print(f"\nDONE! Total canciones: {len(df_songs)}")
display(df_songs)

# Guardar CSV
df_songs.to_csv("songs.csv", index=False)

[1/30] Processing: Taylor Swift
[2/30] Processing: Drake
[3/30] Processing: Morgan Wallen
[4/30] Processing: Kendrick Lamar
[5/30] Processing: The Weeknd
[6/30] Processing: SZA
[7/30] Processing: Zach Bryan
[8/30] Processing: Tyler, The Creator
[9/30] Processing: Billie Eilish
[10/30] Processing: Ariana Grande
[11/30] Processing: Bad Bunny
[12/30] Processing: Karol G
[13/30] Processing: Myke Towers
[14/30] Processing: Quevedo
[15/30] Processing: Aitana
[16/30] Processing: Rosalía
[17/30] Processing: Morad
[18/30] Processing: Melendi
[19/30] Processing: Enrique Iglesias
[20/30] Processing: Juanes
[21/30] Processing: Duki
[22/30] Processing: María Becerra
[23/30] Processing: Tini Stoessel
[24/30] Processing: Nicki Nicole
[25/30] Processing: Cazzu
[26/30] Processing: Lali
[27/30] Processing: Paulo Londra
[28/30] Processing: Shakira
[29/30] Processing: Bizarrap
[30/30] Processing: Emilia

DONE! Total canciones: 1500


,artist_id,artist_name,track_title,album_title,track_type,release_year,genre,genre_id
0,12246,Taylor Swift,The Fate of Ophelia,The Life of a Showgirl,solo,2025,Pop,132
1,12246,Taylor Swift,Opalite,The Life of a Showgirl,solo,2025,Pop,132
2,12246,Taylor Swift,Elizabeth Taylor,The Life of a Showgirl,solo,2025,Pop,132
3,12246,Taylor Swift,Blank Space,1989 (Deluxe),solo,2014,Pop,132
4,12246,Taylor Swift,Out Of The Woods (Taylor's Version),1989 (Taylor's Version),solo,2023,Pop,132
...,...,...,...,...,...,...,...,...
1495,61094422,Emilia,Uno los Dos,Uno los Dos,collab,2023,Pop,132
1496,61094422,Emilia,Diva (feat. Emilia),Diva (feat. Emilia),collab,2022,Rap/Hip Hop,116
1497,61094422,Emilia,Guerrero.mp3,Guerrero.mp3,solo,2023,Pop,132
1498,61094422,Emilia,Somos Más,Somos Más,collab,2026,Pop,132
